### Utility to refresh BCAP node aliases from database.
Generates the files and overwrites existing content in bcap.util.aliases.*

In [ ]:
import os
import django
from dotenv import load_dotenv
import sys

# Add the project root to sys.path so 'bcap' is importable as a package
sys.path.insert(0, "/web_root/bcap")

# Load environment variables from nr-bcap/.env
load_dotenv(dotenv_path=".env")

os.environ.setdefault("DJANGO_SETTINGS_MODULE", "bcap.settings")
django.setup()

In [ ]:
import os
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

from arches.app.models import models


def _get_nodes_for_graph(graph_slug):
    return (models.Node.objects
             .exclude(datatype__in=["semantic"])
             .filter(graph__slug=graph_slug)
             .prefetch_related("graph")).order_by("graph__slug", "alias").all()

def _create_alias_file(graph):
    ### Creates or overwrites the existing file in bcap/util/aliases
    nodes = _get_nodes_for_graph(graph.slug)
    filename = os.path.join("..", "bcap","util","aliases",graph.slug+".py")
    classname = _snake_to_camel(graph.slug)+"Aliases"
    print(graph.slug, filename)
    with open(filename, "w") as alias_file:
        alias_file.write("from bcap.util.bcap_aliases import AbstractAliases\n\n\n")
        alias_file.write(f"class {classname}(AbstractAliases):\n")
        for node in nodes:
            alias_file.write(f"    {node.alias.upper()} = \"{node.alias}\"\n")
        alias_file.write(f"""
    @staticmethod
    def get_aliases():
        return AbstractAliases.get_dict({classname})
""")


def _snake_to_camel(snake_str):
    words = snake_str.split('_')
    return ''.join(word.title() for word in words)

# Driver - change filter if targeting a single resource model
for graph in models.Graph.objects.exclude(slug="arches_system_settings").order_by("slug").all():
    _create_alias_file(graph)
